# Scikit-learn — Part 2 (Classification & Pipelines)

In [ ]:
# Day 28 — Scikit-learn Part 2
# In Day 27 we learned REGRESSION (predicting numbers)
# Today we learn CLASSIFICATION (predicting categories)
#
# Regression: "How much?" → price, temperature, response time
# Classification: "Which one?" → spam/not spam, healthy/critical, cat/dog
#
# Today we'll cover:
#   1. Classification — what and why
#   2. Logistic Regression (despite the name, it's for classification!)
#   3. Decision Trees and Random Forests
#   4. Evaluation: Accuracy, Confusion Matrix, Precision, Recall, F1
#   5. Cross-validation (better than a single train/test split)
#   6. Sklearn Pipelines (chain preprocessing + model in one object)

### Classification — Predicting Categories

In [ ]:
# Let's build a model that predicts if a server needs attention
# Based on: cpu, memory, disk, error_count
# Labels: 'healthy' or 'critical'

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

np.random.seed(42)
n = 300

cpu = np.random.randint(10, 100, n)
memory = np.random.randint(15, 95, n)
disk = np.random.randint(10, 90, n)
error_count = np.random.randint(0, 50, n)

# A server is 'critical' if: high CPU + high memory, or many errors
score = 0.3 * cpu + 0.25 * memory + 0.15 * disk + 0.8 * error_count + np.random.normal(0, 10, n)
label = (score > 50).astype(int)    # 1 = critical, 0 = healthy

df = pd.DataFrame({'cpu': cpu, 'memory': memory, 'disk': disk,
                   'error_count': error_count, 'label': label})

print(df.head(10))
print(f'\nHealthy: {(label == 0).sum()}, Critical: {(label == 1).sum()}')

X = df[['cpu', 'memory', 'disk', 'error_count']]
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Logistic Regression

In [ ]:
# Logistic Regression — for BINARY classification (yes/no, 0/1)
# Despite having 'Regression' in the name, it's a CLASSIFIER!
# It predicts the PROBABILITY of belonging to a class

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Scale features (important for Logistic Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train
lr = LogisticRegression(random_state=42)
lr.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = lr.predict(X_test_scaled)

# Accuracy
accuracy = lr.score(X_test_scaled, y_test)
print(f'Logistic Regression Accuracy: {accuracy:.2%}')

# Predict probabilities
y_proba = lr.predict_proba(X_test_scaled)    # returns [prob_class0, prob_class1]
print('\nFirst 5 predictions with probabilities:')
for i in range(5):
    print(f'  Predicted: {y_pred_lr[i]}, Prob: healthy={y_proba[i][0]:.2f}, critical={y_proba[i][1]:.2f}')

### Decision Tree & Random Forest

In [ ]:
# Decision Tree = a flowchart of yes/no questions
# "Is CPU > 80?" → Yes → "Is error_count > 20?" → Yes → CRITICAL
#
# Random Forest = many decision trees that VOTE together
# Like asking 100 doctors instead of 1 — majority rules!
# Almost always better than a single decision tree

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print(f'Decision Tree Accuracy: {dt.score(X_test, y_test):.2%}')

# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print(f'Random Forest Accuracy: {rf.score(X_test, y_test):.2%}')

# n_estimators = number of trees in the forest
# max_depth = how deep each tree can grow (prevents overfitting)

# Decision Trees & Random Forests DON'T need feature scaling!

In [ ]:
# Feature Importance — which features matter most?

import matplotlib.pyplot as plt

feature_names = X.columns.tolist()
importances = rf.feature_importances_    # how important each feature is

# Sort by importance
sorted_idx = np.argsort(importances)

plt.figure(figsize=(8, 4))
plt.barh(range(len(sorted_idx)), importances[sorted_idx], color='#3498db')
plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
plt.title('Feature Importance (Random Forest)', fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

# This tells you WHICH metrics to monitor most closely!

### Evaluation — Confusion Matrix, Precision, Recall, F1

In [ ]:
# Accuracy alone is NOT enough!
#
# Example: If 95% of servers are healthy and 5% are critical,
# a model that ALWAYS predicts 'healthy' gets 95% accuracy!
# But it misses every critical server — useless!
#
# We need more detailed metrics:
#
# Confusion Matrix:
#                  Predicted Healthy    Predicted Critical
# Actually Healthy      TN                  FP (false alarm)
# Actually Critical     FN (missed!)        TP
#
# Precision = TP / (TP + FP)  -> "Of all predicted critical, how many were truly critical?"
# Recall    = TP / (TP + FN)  -> "Of all actually critical, how many did we catch?"
# F1 Score  = harmonic mean of precision and recall

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Classification report (precision, recall, f1 for each class)
print('=== Random Forest ====')
print(classification_report(y_test, y_pred_rf, target_names=['healthy', 'critical']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
print('Confusion Matrix:')
print(cm)
# [[TN, FP],
#  [FN, TP]]

In [ ]:
# Visualise the confusion matrix

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf,
    display_labels=['Healthy', 'Critical'],
    cmap='Blues', ax=ax
)
plt.title('Confusion Matrix — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

### Cross-Validation

In [ ]:
# Problem with a SINGLE train/test split:
# The result depends on WHICH data ended up in train vs test
# You might get lucky or unlucky with the split
#
# Cross-Validation solves this by doing MULTIPLE splits
#
# 5-Fold CV:
# Split data into 5 parts (folds)
# Train on 4 folds, test on 1 fold — repeat 5 times
# Each fold gets to be the test set once
# Average the 5 scores → more reliable!

from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# 5-fold cross-validation
scores = cross_val_score(rf, X, y, cv=5, scoring='accuracy')

print(f'CV Scores: {scores}')
print(f'Mean Accuracy: {scores.mean():.2%} ± {scores.std():.2%}')

# ± tells you how STABLE the model is
# Small ± = consistent performance
# Large ± = performance varies a lot depending on the data split

### Sklearn Pipelines — Chain Everything Together

In [ ]:
# A Pipeline chains PREPROCESSING + MODEL into one object
# Benefits:
# - No risk of data leakage (scaler won't see test data during fit)
# - Cleaner code (one .fit(), one .predict())
# - Easy to save and deploy

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Create pipeline: scaler → model
pipe = Pipeline([
    ('scaler', StandardScaler()),              # step 1: scale features
    ('model', RandomForestClassifier(n_estimators=100, random_state=42))  # step 2: model
])

# Now use it EXACTLY like a regular model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe.fit(X_train, y_train)           # fits scaler AND model
accuracy = pipe.score(X_test, y_test) # scales AND predicts
print(f'Pipeline Accuracy: {accuracy:.2%}')

# Predict on new data — pipeline handles scaling automatically
new_server = [[85, 90, 70, 25]]    # cpu, memory, disk, errors
prediction = pipe.predict(new_server)
print(f'Prediction for new server: {"CRITICAL" if prediction[0] == 1 else "HEALTHY"}')

# This pipeline object can be SAVED and DEPLOYED as-is
# The person using it doesn't need to know about scaling — it's built in!

In [ ]:
# Practice Questions:

# 1. Build a classifier to predict if a deployment will PASS or FAIL
#    features: code_changes, test_coverage, build_time, is_hotfix
#    Try Logistic Regression, Decision Tree, Random Forest
#    Compare accuracy of all three

# 2. Use classification_report to compare precision and recall
#    Which metric matters more for deployment failure prediction?
#    (Hint: is it worse to miss a failure or have a false alarm?)

# 3. Use cross_val_score with 10 folds to compare models
#    Print mean ± std for each model

# 4. Build a Pipeline with: StandardScaler → RandomForest
#    Train it on the deployment data and use it to predict

### DevOps / AI-MLOps Practice

In [ ]:
# Classification is used EVERYWHERE in monitoring and MLOps

# Exercise 1: Anomaly Detection for Servers
# Create synthetic data for 500 servers:
#   features: cpu, memory, disk, network_in, network_out, error_rate
#   labels: 0=normal, 1=anomaly
# Train a Random Forest classifier
# Print feature importances — which metric best detects anomalies?
# Plot the confusion matrix

# Exercise 2: Model Comparison Report
# Compare 3 models on the same data:
#   LogisticRegression, DecisionTree, RandomForest
# For each model, print:
#   - Accuracy, Precision, Recall, F1
#   - Cross-validation mean ± std
# Create a DataFrame with the comparison results

# Exercise 3: Production Pipeline
# Build a complete Pipeline:
#   StandardScaler → RandomForest
# Use cross_val_score to evaluate
# Print the pipeline steps
# Use the pipeline to predict on 5 new server metrics